# Rehearse release decisions in an offline CI/CD fixture

Evaluate required checks, verify artifact identity, apply a release gate and resolve a compatible rollback target. No credentials or CI runners are needed.

Fill in each starter, then run its check cell. Restart the kernel and run all cells before submitting. Keep your interpretation in the last Markdown cell.


## 1. Find unsatisfied CI checks

Check required job results.

Implement missing_checks(required, statuses). Return sorted unique required names that lack exact success. Missing, skipped and failed all remain unsatisfied.


In [ ]:
def missing_checks(required, statuses):
    raise NotImplementedError


In [ ]:
assert missing_checks(["unit","schema","unit"],{"unit":"success"}) == ["schema"]
assert missing_checks(["a","b"],{"a":"skipped","b":"failure"}) == ["a","b"]
assert missing_checks([],{"other":"failure"}) == []
assert missing_checks(["a","a"],{}) == ["a"]
print("OK")


## 2. Check promoted artifact bytes

Verify artifact digest.

Implement matches_digest(contents, expected) for bytes and a lowercase SHA-256 digest string. Return true only for exact hex digest equality.


In [ ]:
def matches_digest(contents, expected):
    raise NotImplementedError


In [ ]:
from hashlib import sha256
assert matches_digest(b"one",sha256(b"one").hexdigest())
assert not matches_digest(b"two",sha256(b"one").hexdigest())
assert matches_digest(b"",sha256(b"").hexdigest())
assert not matches_digest(b"one","one")
assert not matches_digest(b"one", sha256(b"one").hexdigest()[:12])
assert not matches_digest(b"one", "")
print("OK")


## 3. Collect all promotion blockers

Apply signed-off metric and smoke gates.

Implement gate_failures(ci, smoke, score, floor, critical). Include ci and smoke when the corresponding pass flag is false, quality for missing or below-floor scores, and critical for a critical regression. Preserve this order.


In [ ]:
def gate_failures(ci, smoke, score, floor, critical):
    raise NotImplementedError


In [ ]:
assert gate_failures(True,True,0.9,0.9,False) == []
assert gate_failures(False,False,None,0.9,True) == ["ci","smoke","quality","critical"]
assert gate_failures(True,True,0.89,0.9,False) == ["quality"]
assert gate_failures(True,True,None,0.0,False) == ["quality"]
assert gate_failures(True,True,0.0,0.0,False) == []
print("OK")


## 4. Select a verified rollback target

Find a compatible prior release.

Implement rollback_target(history, current, schema). Each row is (release ID, sequence, feature schema, approved). Return matching approved ID other than current, choosing maximum sequence then ID. Return None if unavailable.


In [ ]:
def rollback_target(history, current, schema):
    raise NotImplementedError


In [ ]:
history=[("a",1,"s",True),("b",2,"s",True),("c",3,"other",True),("d",4,"s",False)]
assert rollback_target(history,"b","s") == "a"
assert rollback_target(history,"current","s") == "b"
assert rollback_target(history,"current","none") is None
assert rollback_target([("a",2,"s",True),("b",2,"s",True)],"new","s") == "b"
print("OK")


## 5. Authorise promotion only for the tested bytes

Implement promotion_report(required, statuses, contents, expected, smoke, score, floor, critical).
Return (missing check names, gate failure names). Gate failures come from gate_failures, with an
additional artifact failure at the end if bytes do not match. A successful report has two empty
lists. This check does not make any deployment changes.


In [ ]:
def promotion_report(required,statuses,contents,expected,smoke,score,floor,critical):
    raise NotImplementedError


In [ ]:
from hashlib import sha256
expected = sha256(b"model").hexdigest()
assert promotion_report(["unit"],{"unit":"success"},b"model",expected,True,0.9,0.9,False) == ([],[])
assert promotion_report(["unit"],{},b"other",expected,False,None,0.9,True) == (["unit"],["ci","smoke","quality","critical","artifact"])
assert promotion_report([],{},b"other",expected,True,1,0.9,False) == ([],["artifact"])
assert promotion_report([],{},b"model-v2",sha256(b"model-v2").hexdigest(),True,1,0.9,False) == ([],[])
assert promotion_report(["unit"],{"unit":"success","lint":"failure"},b"model",expected,True,0.9,0.9,False) == ([],[])
print("OK")


## Interpretation

Draw the jobs and evidence handoffs. Name which failures should prevent deployment and who can decide to restore the previous bundle.

**Your answer**
